# Policy Gradient Visualization - Simple Examples

This notebook gives a very small and concrete introduction to policy gradients using the simplest examples possible.

It covers:

1. a single-state, two-action example,
2. a natural two-state extension,
3. visualizations of parameter evolution and gradient values across iterations.

The goal is not realism. The goal is to make the policy-gradient update visually and mathematically transparent.


## Core idea

In supervised learning, we usually have a target output and minimize a loss such as cross-entropy.

In policy gradient methods, we do something different:

- sample an action or trajectory from the current policy,
- evaluate it with a reward,
- increase the log-probability of actions that led to high reward.

The REINFORCE estimator is

$$
\nabla_\theta J(\theta)
=
\mathbb{E}_{\tau \sim \pi_\theta}
\left[
R(\tau) \, \nabla_\theta \log \pi_\theta(\tau)
\right].
$$

In an autoregressive or sequential setting, the probability of a trajectory factorizes across steps, so one terminal reward can affect the gradient contributions of multiple earlier decisions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


## 1. Single-state, two-action example

We start with the smallest possible policy-gradient system.

There is only one state $s$ and two available actions:

- $a_1$
- $a_2$

We parameterize the policy by a single scalar $\theta$:

$$
\pi_\theta(a_1) = \theta,
\qquad
\pi_\theta(a_2) = 1 - \theta,
\qquad
0 < \theta < 1.
$$

The reward function is deliberately simple:

$$
R(a_1) = 1,
\qquad
R(a_2) = 0.
$$

So action $a_1$ is the good action and action $a_2$ is the bad action.


### Expected reward

The expected reward is

$$
J(\theta)
=
\theta \cdot 1 + (1-\theta) \cdot 0
=
\theta.
$$

Therefore the optimal policy is obvious:

$$
\theta^* = 1.
$$

So the learning problem is simply: can the stochastic policy move its probability mass toward the rewarding action?


### REINFORCE update in this example

The policy-gradient estimator is

$$
\nabla_\theta J(\theta)
=
\mathbb{E}_{a \sim \pi_\theta}
\left[
R(a) \, \nabla_\theta \log \pi_\theta(a)
\right].
$$

The score terms are

$$
\nabla_\theta \log \pi_\theta(a_1) = \frac{1}{\theta},
\qquad
\nabla_\theta \log \pi_\theta(a_2) = -\frac{1}{1-\theta}.
$$

This immediately gives the intuition:

- if we sample $a_1$, the reward is $1$, so the update pushes $\theta$ upward,
- if we sample $a_2$, the reward is $0$, so the update is zero in this minimal setup.

That is the simplest possible visualization of policy gradient: rewarding samples increase their own probability.


### What to look for in the plots

We will visualize two quantities over iterations:

1. $\theta$ itself, which is the probability of choosing the good action,
2. the sampled gradient value at each iteration.

As training proceeds, we expect:

- $\theta$ to drift toward $1$,
- gradient spikes to become smaller and less frequent once the policy has mostly committed to the rewarding action.


In [ ]:
theta = 0.4
lr = 0.05
iterations = 200

theta_history = []
grad_history = []
reward_history = []

for i in range(iterations):
    if np.random.rand() < theta:
        action = 1
        reward = 1
        grad = reward * (1 / theta)
    else:
        action = 2
        reward = 0
        grad = reward * (-1 / (1 - theta))

    theta += lr * grad
    theta = np.clip(theta, 1e-3, 1 - 1e-3)

    theta_history.append(theta)
    grad_history.append(grad)
    reward_history.append(reward)

plt.figure(figsize=(8, 4))
plt.plot(theta_history)
plt.title("Single-state example: parameter $\theta$ over iterations")
plt.xlabel("Iteration")
plt.ylabel("$\theta = \pi(a_1)$")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(grad_history)
plt.title("Single-state example: sampled gradient over iterations")
plt.xlabel("Iteration")
plt.ylabel("Gradient value")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(reward_history)
plt.title("Single-state example: sampled reward over iterations")
plt.xlabel("Iteration")
plt.ylabel("Reward")
plt.show()


### Interpretation of the single-state plots

The first plot shows how the policy parameter evolves. Since $\theta$ is the probability of taking $a_1$, movement upward means the policy is assigning more mass to the rewarding action.

The second plot shows the sampled gradient value. Early in training, when $\theta$ is still modest, sampling $a_1$ produces a relatively large positive gradient:

$$
\frac{1}{\theta}.
$$

As $\theta$ grows, that gradient shrinks, because increasing a probability that is already near $1$ becomes harder.

This is a useful first lesson: policy gradients are not just about reward, but about reward times a score term, and that score term depends on the current policy.


## 2. Two-state extension

Now we extend the example in the most natural way.

We create a trajectory with two decisions:

- one action in state $s_1$,
- one action in state $s_2$.

Each state again has two actions, and each state has its own policy parameter:

$$
\pi_{\theta_1}(a_1 \mid s_1)=\theta_1,
\qquad
\pi_{\theta_1}(a_2 \mid s_1)=1-\theta_1,
$$

$$
\pi_{\theta_2}(a_1 \mid s_2)=\theta_2,
\qquad
\pi_{\theta_2}(a_2 \mid s_2)=1-\theta_2.
$$

The key design choice is this:

- the reward is only determined at the second state.

So this becomes a tiny trajectory-level credit assignment problem.


### Reward structure

We choose:

$$
R(\tau) =
\begin{cases}
1, & \text{if the action at } s_2 \text{ is } a_1, \\
0, & \text{if the action at } s_2 \text{ is } a_2.
\end{cases}
$$

This means the first action does not directly determine reward, but under REINFORCE the same terminal reward is still applied to the gradient terms for both steps:

$$
\nabla_\theta J
=
\mathbb{E}
\left[
R(\tau)
\left(
\nabla \log \pi(a_{s_1} \mid s_1)
+
\nabla \log \pi(a_{s_2} \mid s_2)
\right)
\right].
$$

This is the central idea of trajectory-level policy gradients:

- one reward at the end,
- multiple gradient contributions along the path.


### Why this example is pedagogically useful

In the one-state example, the reward only affected the action that was sampled.

In the two-state example, we can see something more subtle:

- if the second action earns reward,
- then the first action also receives a gradient contribution,
- even though the first state was not directly rewarded.

This is the simplest illustration of delayed credit assignment.


### What to look for in the two-state plots

We will plot:

1. $\theta_1$ and $\theta_2$ across iterations,
2. the sampled gradient contributions for each state.

Typical behavior:

- $\theta_2$ should improve more directly, because reward depends on the second-state decision,
- $\theta_1$ may also move, because the terminal reward multiplies its score term,
- this makes visible how REINFORCE pushes probability mass along whole rewarding trajectories.


In [ ]:
theta1 = 0.5
theta2 = 0.5

lr = 0.05
iterations = 200

theta1_hist = []
theta2_hist = []
grad1_hist = []
grad2_hist = []
reward2_hist = []

for i in range(iterations):
    if np.random.rand() < theta1:
        a1 = 1
        grad1_local = 1 / theta1
    else:
        a1 = 2
        grad1_local = -1 / (1 - theta1)

    if np.random.rand() < theta2:
        a2 = 1
        reward = 1
        grad2_local = 1 / theta2
    else:
        a2 = 2
        reward = 0
        grad2_local = -1 / (1 - theta2)

    g1 = reward * grad1_local
    g2 = reward * grad2_local

    theta1 += lr * g1
    theta2 += lr * g2

    theta1 = np.clip(theta1, 1e-3, 1 - 1e-3)
    theta2 = np.clip(theta2, 1e-3, 1 - 1e-3)

    theta1_hist.append(theta1)
    theta2_hist.append(theta2)
    grad1_hist.append(g1)
    grad2_hist.append(g2)
    reward2_hist.append(reward)

plt.figure(figsize=(8, 4))
plt.plot(theta1_hist, label=r"$\theta_1$ (state 1)")
plt.plot(theta2_hist, label=r"$\theta_2$ (state 2)")
plt.title("Two-state example: policy parameters over iterations")
plt.xlabel("Iteration")
plt.ylabel("Parameter value")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(grad1_hist, label="Gradient from state 1")
plt.plot(grad2_hist, label="Gradient from state 2")
plt.title("Two-state example: sampled gradients with shared terminal reward")
plt.xlabel("Iteration")
plt.ylabel("Gradient value")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(reward2_hist)
plt.title("Two-state example: terminal reward over iterations")
plt.xlabel("Iteration")
plt.ylabel("Reward")
plt.show()


### Interpretation of the two-state plots

The parameter $\theta_2$ usually improves more directly because reward is explicitly tied to the action at the second state.

The interesting quantity is $\theta_1$. Even though state $s_1$ does not itself receive a local reward, its log-probability term is still multiplied by the terminal reward. So when a trajectory ends well, earlier decisions are reinforced too.

This is exactly the mechanism that becomes important in larger problems:

- in CartPole, a final return reinforces many earlier actions,
- in sequence models, a final correctness signal can reinforce many earlier tokens,
- in reasoning models, a terminal answer reward can influence earlier reasoning steps.

So this tiny two-state example is the smallest nontrivial picture of trajectory-level learning.


## Final remarks

This notebook deliberately uses the simplest parameterization possible in order to isolate the policy-gradient mechanism.

The main conceptual lessons are:

1. policy gradient increases the probability of rewarding samples,
2. the update is weighted by a score term $\nabla \log \pi$,
3. in multi-step settings, one terminal reward can propagate backward across several stochastic decisions.

A natural next extension would be to add:

- a baseline, to reduce variance,
- a softmax/logit parameterization,
- a comparison with supervised learning and cross-entropy.
